# 02 · Regime Detection
Train the Gaussian HMM, decode regimes, and explore the results interactively.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt

from src.data_loader import download_price_data, load_close_series
from src.feature_engineering import build_feature_matrix
from src.hmm_model import (
    build_hmm, fit_hmm, predict_states,
    compute_state_statistics, label_states,
    build_regime_dataframe, print_model_diagnostics,
    save_model,
)
from src.visualization import (
    plot_closing_price, plot_price_by_regime,
    plot_return_histograms, plot_transition_matrix,
)
from src.utils import configure_logging, resolve_path, print_state_statistics

configure_logging()
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
TICKER = 'SPY'
START  = '2010-01-01'
END    = '2026-06-01'

raw   = download_price_data(TICKER, START, END, resolve_path('data', 'raw'))
close = load_close_series(raw)
log_returns, X, scaler = build_feature_matrix(close)
print(f'Feature matrix: {X.shape}')

In [ ]:
model = build_hmm()
model = fit_hmm(model, X)
states = predict_states(model, X)
print_model_diagnostics(model)

In [ ]:
stats     = compute_state_statistics(log_returns, states)
labels    = label_states(stats)
regime_df = build_regime_dataframe(log_returns, states, labels)
print_state_statistics(stats, labels)

In [ ]:
fig = plot_closing_price(close, TICKER)
plt.show()

In [ ]:
fig = plot_price_by_regime(close, regime_df, labels, TICKER)
plt.show()

In [ ]:
fig = plot_return_histograms(regime_df, labels)
plt.show()

In [ ]:
fig = plot_transition_matrix(model.transmat_, labels)
plt.show()

In [ ]:
save_model(model, resolve_path('outputs', 'models', f'hmm_{TICKER.lower()}.pkl'))
regime_df.to_csv(resolve_path('outputs', f'regime_labels_{TICKER.lower()}.csv'))
print('Artefacts saved.')